# Writing a finite element nonlinearity

MORFE never sees a mesh. It needs the nonlinear terms $\mathbf T_j$ of the full-order model only as functions it can evaluate on given vectors. This notebook writes the geometric nonlinearity of the clamped–clamped beam of `from_a_mesh_to_a_rom` ourselves, in two ways:

1. as a **closure** that assembles a global vector, the quickest way to plug any finite element code into MORFE;
2. through MORFE's **element-level interface**, which hands the mesh loop to MORFE, so that it can **batch** all the evaluations a monomial needs into one pass over the mesh.

Both reproduce the ROM of MORFEFerrite's built-in term to round-off, but at order 9 the second traverses the mesh 28 times instead of 486.

Before the first run, run `julia setup.jl` in the bash from this repository. For more information look at the README.md of the repository.

In [ ]:
using MORFE, MORFEFerrite, Ferrite
const SVK = MORFEFerrite.StructuralSVK # for Saint-Venant-Kirchhoff hyperelasticity

## 1. Borrow the mesh and the linear part

We write only the nonlinear terms. The mesh, the clamped ends, the stiffness and mass matrices and the spectral data come from the structural backend, exactly as in `from_a_mesh_to_a_rom`, and the backend's own `model` stays as the reference our terms must reproduce. `case.info` exposes the three Ferrite objects we need: the `DofHandler`, the `CellValues` with the shape functions and the quadrature rule, and the map from global to free degrees of freedom, which leaves out the clamped ones.

They are declared `const` because the functions below read them as globals: a non-constant global makes every call type-unstable and the assembly many times slower.

In [ ]:
order = 9
case = SVK.mechanical_model(
    joinpath(@__DIR__, "..", "from_a_mesh_to_a_rom", "clamped_clamped_beam.msh");
    material = SVK.SVKMaterial(E = 160e3, ν = 0.22, ρ = 2.32e-3),
    damping = SVK.RayleighDamping(α = 0.0, β = 0.0),
    dirichlet = "Dirichlet")
(; model, spectral) = build_model(case; master = [1], expansion_order = order)

const dh = case.info.dh                # degrees of freedom of every cell
const cv = case.info.cellvalues        # shape functions and quadrature rule
const free = case.info.free_to_local   # global dof => free dof, clamped ones absent
const λ, μ = case.material.λ, case.material.μ
model.n_fom # print number of free degrees of freedom

## 2. The physics at one quadrature point

For a St. Venant–Kirchhoff material, the Green–Lagrange strain and the second and first Piola–Kirchhoff stresses are

$$\mathbf E = \boldsymbol\varepsilon(\mathbf u) + \tfrac12 \nabla\mathbf u^\top \nabla\mathbf u \,, \qquad \mathbf S = \lambda \operatorname{tr}(\mathbf E)\,\mathbf I + 2\mu\mathbf E \,, \qquad \mathbf P = (\mathbf I + \nabla\mathbf u)\,\mathbf S \,,$$

and the internal force, tested with a virtual displacement $\delta\mathbf u$, is $\int_\Omega \mathbf P : \nabla\delta\mathbf u \, d\Omega$. The stiffness matrix already holds the part of $\mathbf P$ linear in $\mathbf u$; the rest is quadratic and cubic,

$$\mathbf P_2 = \mathbf S\bigl(\tfrac12\nabla\mathbf u^\top\nabla\mathbf u\bigr) + \nabla\mathbf u\,\mathbf S\bigl(\boldsymbol\varepsilon(\mathbf u)\bigr) \,, \qquad \mathbf P_3 = \nabla\mathbf u\,\mathbf S\bigl(\tfrac12\nabla\mathbf u^\top\nabla\mathbf u\bigr) \,.$$

MORFE evaluates each term on different coefficient vectors at once, so we **polarise** it: `P2(a, b)` and `P3(a, b, c)` below are symmetric in the gradients $\mathbf a, \mathbf b, \mathbf c$ and reduce to the expressions above when all gradients are equal. The minus sign moves them to the right-hand side, where MORFE expects the nonlinear terms. The gradients are complex and nothing may conjugate them: conjugation is not linear and would break the method. The code writes `transpose` explicitly; on ordinary Julia matrices `'` conjugates, although Tensors.jl defines it as a plain transpose.

In [ ]:
# S as a function of a strain, the linear strain and the polarised ½∇uᵀ∇u
σ(E) = λ * tr(E) * one(E) + 2μ * E
ε(a) = symmetric(a)
E2(a, b) = symmetric(transpose(a) ⋅ b + transpose(b) ⋅ a) / 4
# the quadratic and cubic parts of the first Piola–Kirchhoff stress, polarised
P2(a, b) = -(σ(E2(a, b)) + (a ⋅ σ(ε(b)) + b ⋅ σ(ε(a))) / 2)
P3(a, b, c) = -(a ⋅ σ(E2(b, c)) + b ⋅ σ(E2(a, c)) + c ⋅ σ(E2(a, b))) / 3;

## 3. Route one: a closure

The simplest finite element term is a function `f!(res, u1, u2)` that adds $\mathbf T(\mathbf u_1, \mathbf u_2)$ to `res`, like the analytical terms of `building_a_full-order_model`. Its body here is an ordinary Ferrite assembly: for every cell, gather the cell values of each argument, integrate $\nabla\delta\mathbf u : \mathbf P$ over the quadrature points and add the cell vector to `res`. Nothing in it belongs to MORFE, so any finite element library fits the same mould.

`gather!` and `scatter!` move values between a free vector and one cell, with zeros for the clamped degrees of freedom. `multiindex = (2, 0)` says that the quadratic term takes two displacements and no velocity, and `fully_asymmetric = false` declares it symmetric in them, so MORFE evaluates each unordered combination of coefficients once. A counter records how often the mesh is traversed.

In [ ]:
const ue = [zeros(ComplexF64, ndofs_per_cell(dh)) for _ in 1:3] # values of 3 arguments
const Fe = zeros(ComplexF64, ndofs_per_cell(dh))                # cell vector
const passes = Ref(0)                                           # mesh traversals

function gather!(ue, u, cell)
    for (i, d) in enumerate(celldofs(cell))
        j = get(free, d, 0)
        ue[i] = j == 0 ? 0 : u[j]
    end
end

function scatter!(res, Fe, cell)
    for (i, d) in enumerate(celldofs(cell))
        j = get(free, d, 0)
        j == 0 || (res[j] += Fe[i])
    end
end

function assemble!(res, P, us::Vararg{Any, N}) where {N}
    passes[] += 1
    for cell in CellIterator(dh)
        reinit!(cv, cell)
        for k in 1:N
            gather!(ue[k], us[k], cell)
        end
        fill!(Fe, 0)
        for q in 1:getnquadpoints(cv)
            Pq = P(ntuple(k -> function_gradient(cv, q, ue[k]), N)...)
            dΩ = getdetJdV(cv, q)
            for i in eachindex(Fe)
                Fe[i] += (shape_gradient(cv, q, i) ⊡ Pq) * dΩ
            end
        end
        scatter!(res, Fe, cell)
    end
    return res
end

quadratic = MultilinearMap((res, u1, u2) -> assemble!(res, P2, u1, u2);
    multiindex = (2, 0), multiplicity_external = 0, fully_asymmetric = false)
cubic = MultilinearMap((res, u1, u2, u3) -> assemble!(res, P3, u1, u2, u3);
    multiindex = (3, 0), multiplicity_external = 0, fully_asymmetric = false)
closure_model = NthOrderModel(case.B, (quadratic, cubic))

`case.B` holds the backend's matrices $(\mathbf K, \mathbf C, \mathbf M)$. The spectral data depend only on them, so `closure_model` reuses `spectral` unchanged. A first solve at order 3 compiles everything, so that the timed solve measures work rather than compilation.

In [ ]:
resonance = ResonanceConfig(style = :complex_normal_form, tol = 0.05)
parametrise(closure_model, spectral, 3; resonance);   # compile
passes[] = 0
t_closure = @elapsed (W_closure, R_closure) =
    parametrise(closure_model, spectral, order; resonance)
(mesh_passes = passes[], seconds = t_closure) # print mesh traversals and solve time

## 4. Why the closure is slow

A coefficient of degree $p$ collects a contribution from every way of writing its monomial as a product of lower-degree ones, its *factorisations*. The cubic term, for instance, reaches $z^5\bar z^4$ through the coefficients of $z$, $z^2\bar z^2$ and $z^2\bar z^2$, and through every other triple of exponents that adds up to $(5, 4)$. MORFE calls the closure once per factorisation, and every call traverses the whole mesh, gathers the same few coefficients again and recomputes their gradients. At order 9 that makes 486 traversals for the 28 monomials of degree 2 to 9 that MORFE solves (it fills their complex conjugates for free), although the factorisations of one monomial share only a few distinct coefficients.

## 5. Route two: the element-level interface

Batching needs MORFE to run the mesh loop, so the term must describe one cell at a time. It becomes a subtype of `FEMMultilinearMap`, carries the same four fields as a `MultilinearMap`, and extends a handful of primitives:

- `fem_elements` and `fem_reinit!` iterate over the cells and prepare each one;
- `fem_n_qp`, `fem_ndofs_per_cell` and `fem_getdetJdV` give the quadrature points, the cell size and the integration weights;
- `scatter_qp!` computes the gradient of one coefficient at every quadrature point of a cell;
- `accumulate_qp!` adds the contribution of one factorisation at one quadrature point to the cell vector, where `mult` counts the orderings of the factorisation;
- `assemble_element!` adds the cell vector to the global one;
- `fem_qp_buffer` returns a buffer whose element type tells MORFE what `scatter_qp!` produces. Spell it out in full, `Tensor{2, 3, ComplexF64, 9}`: without the last parameter the type is abstract, and every gradient stored in the buffer is boxed.

MORFE then traverses the mesh once per monomial for all terms together. In each cell it calls `scatter_qp!` once per distinct coefficient, `accumulate_qp!` once per factorisation and quadrature point, and `assemble_element!` once per term. The physics stays the same two lines, `P2` and `P3`; only the bookkeeping moves.

In [ ]:
struct SVKTerm{DEG} <: FEMMultilinearMap{2}
    multiindex::NTuple{2, Int}
    multiplicity_external::Int
    deg::Int
    fully_asymmetric::Bool
end
SVKTerm(deg) = SVKTerm{deg}((deg, 0), 0, deg, false)
stress(::SVKTerm{2}) = P2
stress(::SVKTerm{3}) = P3

MORFE.fem_elements(::SVKTerm) = (passes[] += 1; CellIterator(dh))
MORFE.fem_reinit!(cell, ::SVKTerm) = reinit!(cv, cell)
MORFE.fem_n_qp(::SVKTerm) = getnquadpoints(cv)
MORFE.fem_ndofs_per_cell(::SVKTerm) = ndofs_per_cell(dh)
MORFE.fem_getdetJdV(cell, q, ::SVKTerm) = getdetJdV(cv, q)
MORFE.fem_qp_buffer(::SVKTerm{DEG}) where {DEG} =
    Matrix{Tensor{2, 3, ComplexF64, 9}}(undef, DEG, getnquadpoints(cv))

function MORFE.scatter_qp!(∇u, u, cell, ::SVKTerm)
    gather!(ue[1], u, cell)
    for q in eachindex(∇u)
        ∇u[q] = function_gradient(cv, q, ue[1])
    end
end

function MORFE.accumulate_qp!(Fe, ∇u, mult, cell, q, dΩ, t::SVKTerm)
    Pq = stress(t)(∇u...)
    for i in eachindex(Fe)
        Fe[i] += mult * (shape_gradient(cv, q, i) ⊡ Pq) * dΩ
    end
end

MORFE.assemble_element!(res, Fe, cell, ::SVKTerm) = scatter!(res, Fe, cell)

element_model = NthOrderModel(case.B, (SVKTerm(2), SVKTerm(3)))

The same `parametrise` call solves it, and `fem_elements` now counts the traversals.

In [ ]:
parametrise(element_model, spectral, 3; resonance);   # compile
passes[] = 0
t_element = @elapsed (W_element, R_element) =
    parametrise(element_model, spectral, order; resonance)
(mesh_passes = passes[], seconds = t_element) # print mesh traversals and solve time

## 6. Same ROM, fewer traversals

All three models share one `spectral`, so their eigenvectors, and with them the gauge of every coefficient, are identical and the coefficients compare directly. The two routes agree with each other, and with the backend's built-in term, which arranges the same formula differently, to round-off.

In [ ]:
W_ref, R_ref = parametrise(model, spectral, order; resonance)
rel(a, b) = maximum(abs, a.poly.coefficients - b.poly.coefficients) /
            maximum(abs, b.poly.coefficients)
(W_closure_vs_element = rel(W_closure, W_element),
    W_element_vs_builtin = rel(W_element, W_ref),
    R_element_vs_builtin = rel(R_element, R_ref)) # print relative differences

## 7. Where the time goes

Both solves also factorise one sparse matrix per monomial, at the same cost, so the solve times above understate the difference. MORFE's benchmark mode times the right-hand side on its own: `solve_parametrisation` with a `benchmark_dir` writes the time spent per degree. Timings are noisy, so each point keeps the fastest of three runs.

In [ ]:
using MORFE.Resonance: build_resonance_set

function rhs_cost_once(m, p)
    mset = build_multiindex_set(p, 2)
    dir = mktempdir()
    passes[] = 0
    rset = build_resonance_set(m, mset, spectral, resonance)
    quiet = ParametrisationOptions(show_progress = false, verbose = false)
    solve_parametrisation(m, mset, spectral, rset; benchmark_dir = dir, options = quiet)
    rows = split.(readlines(joinpath(dir, "benchmark_per_order.csv"))[2:end], ',')
    return (passes = passes[], seconds = sum(r -> parse(Float64, r[3]), rows))
end
rhs_cost(m, p) = argmin(c -> c.seconds, [rhs_cost_once(m, p) for _ in 1:3]) # least noisy of 3

orders = 3:2:order
closure_cost = [rhs_cost(closure_model, p) for p in orders]
element_cost = [rhs_cost(element_model, p) for p in orders]

The traversals of the closure grow with the number of factorisations, those of the element-level term only with the number of monomials. The right-hand-side time falls by less than the traversals, because `accumulate_qp!` still runs once per factorisation: batching shares the traversal, the gathering and the gradients, not the physics. The gain therefore grows with the order and with the number of reduced coordinates, and it is largest for cheap integrands.

In [ ]:
using CairoMakie
fig = Figure(size = (720, 320))
ax1 = Axis(fig[1, 1]; xlabel = "order", ylabel = "mesh traversals", xticks = orders,
    yscale = log10, yticks = [3, 10, 30, 100, 300])
ax2 = Axis(fig[1, 2]; xlabel = "order", ylabel = "right-hand-side time  [s]",
    xticks = orders)
for (cost, label) in ((closure_cost, "closure"), (element_cost, "element-level"))
    scatterlines!(ax1, orders, [c.passes for c in cost]; label)
    scatterlines!(ax2, orders, [c.seconds for c in cost]; label)
end
axislegend(ax1; position = :lt)
fig

## 8. Save the cost

The last cell writes these costs to `results/data/rhs_cost.csv`, the data the tutorial page draws.

In [ ]:
mkpath(joinpath(@__DIR__, "results", "data"))
open(joinpath(@__DIR__, "results", "data", "rhs_cost.csv"), "w") do io
    println(io, "order,passes_closure,passes_element,rhs_closure_s,rhs_element_s")
    for (p, c, e) in zip(orders, closure_cost, element_cost)
        println(io, join((p, c.passes, e.passes, c.seconds, e.seconds), ","))
    end
end

## Where next

`from_a_mesh_to_a_rom` solves the same beam with MORFEFerrite's built-in version of this term, which also accepts anisotropic materials. `benchmark.jl` in this folder repeats the comparison of section 7 with that production kernel.